# Compute EmbeddingGemma 2 product embeddings

Replaces the fine-tuned (DoubleMLDeep) product embeddings with off-the-shelf embeddings from
[EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) (`google/embeddinggemma-2`, 740M parameters, 768-dimensional output with Matryoshka truncation to 512/256/128).

* One embedding per product (ASIN): the product text and image are time-invariant.
* `txtimg`: product text and product image interleaved into a single multimodal embedding (counterpart of the text+image model used in the main analysis).
* `txt`: product text only.
* Clustering task prefix `task: clustering | query: ` (symmetric task, see model card), mean pooling, L2-normalized, default vision budget of 280 soft tokens per image.
* Inference in `bfloat16` (recommended by the model card on hardware with native bf16 support; the CPU used here supports AMX-BF16).

The full 768-dimensional embeddings are stored in `../data/embeddings_gemma/`; Matryoshka truncation is applied when building the datasets (`01_4_create_dataset_gemma.ipynb`).

In [ ]:
import numpy as np
import pandas as pd

from utils.utils_gemma import (
    load_product_texts,
    load_product_image_bytes,
    compute_gemma_embeddings,
    embedding_metadata,
    save_embeddings,
)

dtype = "bfloat16"
num_threads = 4
data_dir = "../data"
output_dir = "../data/embeddings_gemma"
cache_dir = "../data/embeddings_gemma/_cache"

dataset_paths = [
    f"{data_dir}/dataset_txt_only_False_embeddings_True_train.zip",
    f"{data_dir}/dataset_txt_only_False_embeddings_True_val.zip",
]
image_dir = f"{data_dir}/amzn_toys_monthly_avg_long_ffill_28_7_images"

## Products

In [ ]:
df_products = load_product_texts(dataset_paths)
print(f"Number of products: {len(df_products)}")
print(df_products["text"].str.len().describe())
df_products.head()

In [ ]:
image_bytes = load_product_image_bytes(image_dir, df_products["ASIN"])
print(f"Number of product images: {len(image_bytes)}")

## Text-only embeddings

In [ ]:
df_emb_txt = compute_gemma_embeddings(
    df_products,
    modality="txt",
    dtype=dtype,
    batch_size=8,
    cache_dir=cache_dir,
    num_threads=num_threads,
)
save_embeddings(
    df_emb_txt,
    f"{output_dir}/embeddinggemma2_txt.parquet",
    metadata=embedding_metadata("txt", dtype, len(df_emb_txt)),
)
print(df_emb_txt.shape)

## Text + image embeddings

In [ ]:
df_emb_txtimg = compute_gemma_embeddings(
    df_products,
    modality="txtimg",
    image_bytes=image_bytes,
    dtype=dtype,
    batch_size=8,
    cache_dir=cache_dir,
    num_threads=num_threads,
)
save_embeddings(
    df_emb_txtimg,
    f"{output_dir}/embeddinggemma2_txtimg.parquet",
    metadata=embedding_metadata("txtimg", dtype, len(df_emb_txtimg)),
)
print(df_emb_txtimg.shape)

## Sanity checks

In [ ]:
for name, df_emb in [("txt", df_emb_txt), ("txtimg", df_emb_txtimg)]:
    norms = np.linalg.norm(df_emb.values, axis=1)
    print(
        f"{name}: shape={df_emb.shape}, finite={np.isfinite(df_emb.values).all()}, "
        f"norm min/max={norms.min():.6f}/{norms.max():.6f}"
    )

cos_txt_txtimg = (df_emb_txt.values * df_emb_txtimg.loc[df_emb_txt.index].values).sum(axis=1)
print("Cosine similarity text vs. text+image embedding of the same product:")
print(pd.Series(cos_txt_txtimg).describe())

In [ ]:
# Nearest neighbours (text+image embedding) of a few products
rng = np.random.default_rng(42)
texts = df_products.set_index("ASIN")["text"]
for asin in rng.choice(df_emb_txtimg.index, size=3, replace=False):
    sims = df_emb_txtimg.values @ df_emb_txtimg.loc[asin].values
    nn = df_emb_txtimg.index[np.argsort(-sims)[1:4]]
    print(f"{asin}: {texts[asin][:100]}")
    for n in nn:
        print(f"    -> {n}: {texts[n][:100]}")